# Comtrade Extraction — 2015-2016 and 2024-2025, in one fail-safe run

Processes both ranges in a loop -- edit `RUNS` in the config cell if filenames change, otherwise
just run top to bottom. Each run is wrapped so one range's failure doesn't stop the other.

**Every issue actually hit while building this, now handled automatically:**
- `encoding='latin-1'` + `index_col=False` on every read -- accented characters and a
  trailing-empty-field mismatch both crashed/misaligned earlier attempts.
- HS matching via `cmdCode.str[:4]` -- works whether the source uses 4-digit or 6-digit codes.
- **`partnerCode == '0'` (World) is now dropped explicitly**, not left to `DROP_CODES` alone --
  both raw exports turned out to include a World-total row per reporter/product/year
  alongside the real bilateral rows, and silently keeping it would double-count trade value
  as if World were a real trading partner.
- **`isAggregate` is checked tolerant of `'0'`/`'false'`/`'true'` variants**, and if a file's
  *actual bilateral rows* (i.e. after the World-row is already dropped) turn out to be 100%
  `isAggregate=true` -- which happened for the 2024-2025 pull, likely because very recent data
  hasn't reached Comtrade's fully-validated status yet -- the cleaner **automatically falls
  back** to accepting `isAggregate=true` for that file rather than silently producing zero
  rows, and prints a loud warning so this gets flagged as a methodology caveat in the paper
  (these years may reflect preliminary/provisional Comtrade figures, not final validated ones).
- Per-run try/except -- if one file is missing or malformed, you still get the other one's
  output instead of the whole notebook dying.

In [4]:
import pandas as pd
import numpy as np
import os
import sys
import pyarrow as pa
import pyarrow.parquet as pq

RAW_DIR = os.path.join("..", "..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "..", "data", "processed")
sys.path.append(os.path.join("..", "..", "src", "utils"))
from country_codes import DROP_CODES

CHIP_HEADINGS = ["8541", "8542"]   # match your paper's stated scope; add "8517" for the broader electronics definition
CHUNK = 200_000
READ_CSV_KW = dict(encoding="latin-1", index_col=False)

_KEEP = [
    "refYear", "reporterCode", "partnerCode", "cmdCode", "flowCode",
    "primaryValue", "FOBValue", "netWgt", "isAggregate",
    "pop_o", "gdp_o", "gdpcap_o", "pop_d", "gdp_d", "gdpcap_d", "dist",
]
_NUMERIC = ["primaryValue", "FOBValue", "netWgt",
            "pop_o", "gdp_o", "gdpcap_o", "pop_d", "gdp_d", "gdpcap_d", "dist"]

# ============================================================
# ---- the only cell that should need editing between runs ----
RUNS = [
    dict(tag="2015_2016", src=os.path.join(RAW_DIR, "comtrade_2015_2016.csv"), year_lo=2015, year_hi=2016),
    dict(tag="2024_2025", src=os.path.join(RAW_DIR, "comtrade_2024_2025.csv"), year_lo=2024, year_hi=2025),
]
# ============================================================
os.makedirs(PROCESSED_DIR, exist_ok=True)

## Step functions -- extract (year+HS filtered), then clean (aggregate/World/zero-value rules)

In [5]:
def extract_yearfiltered_hsfiltered(src, out, year_lo, year_hi, headings=CHIP_HEADINGS, chunk=CHUNK):
    keep = [c for c in _KEEP if c in pd.read_csv(src, nrows=0, **READ_CSV_KW).columns]
    writer, total, kept = None, 0, 0
    for i, ch in enumerate(pd.read_csv(src, usecols=keep, chunksize=chunk,
                                       dtype=str, low_memory=False, **READ_CSV_KW), 1):
        total += len(ch)
        yr = pd.to_numeric(ch["refYear"], errors="coerce")
        code_col = ch["cmdCode"].astype(str)
        mask = yr.between(year_lo, year_hi) & code_col.str[:4].isin(headings)
        ch = ch[mask]
        if ch.empty:
            continue
        for c in _NUMERIC:
            if c in ch: ch[c] = pd.to_numeric(ch[c], errors="coerce")
        ch["chapter"], ch["heading"] = code_col.str[:2], code_col.str[:4]
        kept += len(ch)
        table = pa.Table.from_pandas(ch, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(out, table.schema)
        writer.write_table(table)
    if writer:
        writer.close()
    print(f"  extracted: kept {kept:,} of {total:,} scanned")
    return kept


def _isagg_mask(series, accept_aggregate=False):
    '''True where a row should be KEPT. accept_aggregate=False -> only isAggregate in {0,false}.
    accept_aggregate=True -> keep everything regardless of the flag (used as a fallback).'''
    if accept_aggregate:
        return pd.Series(True, index=series.index)
    norm = series.astype(str).str.strip().str.lower()
    return norm.isin(["0", "false"])


def clean_slice(src, out, accept_aggregate=False):
    pf = pq.ParquetFile(src)
    writer, total, kept = None, 0, 0
    for b in range(pf.num_row_groups):
        ch = pf.read_row_group(b).to_pandas()
        total += len(ch)
        ch = ch[ch["partnerCode"].astype(str) != "0"]              # drop World-total rows explicitly
        ch = ch[_isagg_mask(ch["isAggregate"], accept_aggregate)]   # aggregate flag, with fallback support
        ch = ch[~ch["reporterCode"].astype(str).isin(DROP_CODES)]
        ch = ch[~ch["partnerCode"].astype(str).isin(DROP_CODES)]
        ch["primaryValue"] = pd.to_numeric(ch["primaryValue"], errors="coerce")
        ch = ch[ch["primaryValue"] > 0]
        if ch.empty:
            continue
        kept += len(ch)
        table = pa.Table.from_pandas(ch, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(out, table.schema)
        writer.write_table(table)
    if writer:
        writer.close()
    return kept, total

## Run both ranges -- each wrapped so one failure doesn't block the other

In [6]:
results = []

for run in RUNS:
    tag, src, year_lo, year_hi = run["tag"], run["src"], run["year_lo"], run["year_hi"]
    print(f"\n{'='*60}\n{tag}  (source: {src})\n{'='*60}")

    if not os.path.exists(src):
        print(f"  SKIPPED -- file not found: {src}")
        results.append(dict(tag=tag, status="missing_source"))
        continue

    try:
        out_slice = os.path.join(PROCESSED_DIR, f"all_products_{tag}.parquet")
        out_ready = os.path.join(PROCESSED_DIR, f"all_products_{tag}_ready.parquet")

        kept_rows = extract_yearfiltered_hsfiltered(src, out_slice, year_lo, year_hi)
        if kept_rows == 0:
            print(f"  SKIPPED -- no rows matched years {year_lo}-{year_hi} / headings {CHIP_HEADINGS}")
            results.append(dict(tag=tag, status="no_rows_extracted"))
            continue

        # sanity check on the raw slice
        pf = pq.ParquetFile(out_slice)
        s = pf.read_row_group(0).to_pandas()
        years_seen = sorted(pd.to_numeric(s["refYear"], errors="coerce").dropna().unique().tolist())
        n_partners = s["partnerCode"].nunique()
        print(f"  years in sample: {years_seen}  |  distinct partners in sample: {n_partners}")
        if n_partners <= 1:
            print("  WARNING: only one partner code seen -- this may still be a World-total-only export.")

        # clean, strict first
        kept_strict, total_clean = clean_slice(out_slice, out_ready, accept_aggregate=False)
        used_fallback = False
        if kept_strict == 0 and total_clean > 0:
            print("  strict isAggregate filter kept 0 rows -- trying fallback (accept_aggregate=True)")
            kept_fallback, total_fallback = clean_slice(out_slice, out_ready, accept_aggregate=True)
            if kept_fallback > 0:
                kept_strict, total_clean, used_fallback = kept_fallback, total_fallback, True
            else:
                # fallback didn't help either -- the real blocker is something else (e.g. every
                # row was a World-total row, dropped before isAggregate was ever checked)
                print("  fallback ALSO kept 0 rows -- isAggregate wasn't the actual blocker. "
                      "Check whether every row has partnerCode=='0' (World-only export).")

        pct = 100 * kept_strict / max(total_clean, 1)
        print(f"  cleaned: kept {kept_strict:,} of {total_clean:,} ({pct:.1f}%)"
              + ("  [FALLBACK: isAggregate=true accepted]" if used_fallback else ""))
        print(f"  -> {out_ready}")

        results.append(dict(tag=tag, status="ok", extracted=kept_rows, cleaned=kept_strict,
                            years=years_seen, fallback_used=used_fallback, out=out_ready))
    except Exception as e:
        print(f"  ERROR: {type(e).__name__}: {e}")
        results.append(dict(tag=tag, status="error", error=str(e)))


2015_2016  (source: ../../data/raw/comtrade_2015_2016.csv)
  extracted: kept 24,008 of 24,008 scanned
  years in sample: [2015, 2016]  |  distinct partners in sample: 240
  cleaned: kept 10,147 of 24,008 (42.3%)
  -> ../../data/processed/all_products_2015_2016_ready.parquet

2024_2025  (source: ../../data/raw/comtrade_2024_2025.csv)
  extracted: kept 21,731 of 21,731 scanned
  years in sample: [2024, 2025]  |  distinct partners in sample: 241
  strict isAggregate filter kept 0 rows -- trying fallback (accept_aggregate=True)
  cleaned: kept 21,317 of 21,731 (98.1%)  [FALLBACK: isAggregate=true accepted]
  -> ../../data/processed/all_products_2024_2025_ready.parquet


## Summary

In [7]:
summary = pd.DataFrame(results)
print(summary.to_string(index=False))

fallback_tags = [r["tag"] for r in results if r.get("fallback_used")]
if fallback_tags:
    print(f"\nNOTE FOR THE PAPER: {fallback_tags} required accepting isAggregate=true rows "
          f"(their genuinely bilateral rows never had isAggregate=false/0) -- flag these as "
          f"preliminary/provisional Comtrade figures rather than final validated data.")

failed = [r["tag"] for r in results if r["status"] != "ok"]
if failed:
    print(f"\nNeeds attention: {failed} -- see the per-run output above for the specific reason.")
else:
    print("\nBoth ranges completed successfully.")

      tag status  extracted  cleaned        years  fallback_used                                                       out
2015_2016     ok      24008    10147 [2015, 2016]          False ../../data/processed/all_products_2015_2016_ready.parquet
2024_2025     ok      21731    21317 [2024, 2025]           True ../../data/processed/all_products_2024_2025_ready.parquet

NOTE FOR THE PAPER: ['2024_2025'] required accepting isAggregate=true rows (their genuinely bilateral rows never had isAggregate=false/0) -- flag these as preliminary/provisional Comtrade figures rather than final validated data.

Both ranges completed successfully.
